Silver overview

## Silver Layer (Clean & Feature-Engineered)

**Goal:** Clean the raw data and create analysis-ready features.  
This is where we fix data types and create new columns like churn_flag and tenure buckets.

Create silver schema 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;


 Read Bronze Table

In [0]:
df = spark.table("bronze.telco_churn_raw")
display(df.limit(10))
df.printSchema()


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: long (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



Clean TotalCharges

TotalCharges is often stored as a string and sometimes contains blanks (" ").
We convert blank values into NULL.
Then we safely cast valid values into double (numeric).
This prevents casting errors later in SQL and Power BI.

In [0]:
from pyspark.sql.functions import col, when, trim

df_clean = (
    df
    .withColumn(
        "TotalCharges",
        when(trim(col("TotalCharges")) == "", None)
        .otherwise(col("TotalCharges").cast("double"))
    )
)


Create churn_flag and senior_citizen_flag

Converts Churn from text ("Yes"/"No") into numeric:
Yes → 1
No → 0.  
 Keeps SeniorCitizen as a numeric flag but gives it a clearer name.
Numeric flags make churn calculations easier and prevent Power BI issues.

In [0]:
from pyspark.sql.functions import when

df_clean = (
    df_clean
    .withColumn("churn_flag", when(col("Churn") == "Yes", 1).otherwise(0))
    .withColumn("senior_citizen_flag", col("SeniorCitizen"))
)


Tenure buckets

Groups customers into lifecycle buckets based on tenure.
Helps identify where churn is highest (early vs mature customers).
Makes visuals more readable than using raw tenure numbers.

In [0]:
df_clean = (
    df_clean
    .withColumn(
        "tenure_bucket",
        when(col("tenure") <= 12, "0–12 months")
        .when(col("tenure") <= 24, "13–24 months")
        .when(col("tenure") <= 48, "25–48 months")
        .otherwise("49+ months")
    )
)


Select final columns for silver

Keeps only the columns needed for churn analysis.
Removes extra columns to make the dataset clean and BI-friendly.
Ensures consistent schema for downstream SQL metrics.

In [0]:
df_silver = df_clean.select(
    "customerID",
    "gender",
    "senior_citizen_flag",
    "Partner",
    "Dependents",
    "tenure",
    "tenure_bucket",
    "Contract",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "churn_flag"
)


In [0]:
display(df_silver.limit(10)) 

customerID,gender,senior_citizen_flag,Partner,Dependents,tenure,tenure_bucket,Contract,PaymentMethod,MonthlyCharges,TotalCharges,churn_flag
7590-VHVEG,Female,0,Yes,No,1,0–12 months,Month-to-month,Electronic check,29.85,29.85,0
5575-GNVDE,Male,0,No,No,34,25–48 months,One year,Mailed check,56.95,1889.5,0
3668-QPYBK,Male,0,No,No,2,0–12 months,Month-to-month,Mailed check,53.85,108.15,1
7795-CFOCW,Male,0,No,No,45,25–48 months,One year,Bank transfer (automatic),42.3,1840.75,0
9237-HQITU,Female,0,No,No,2,0–12 months,Month-to-month,Electronic check,70.7,151.65,1
9305-CDSKC,Female,0,No,No,8,0–12 months,Month-to-month,Electronic check,99.65,820.5,1
1452-KIOVK,Male,0,No,Yes,22,13–24 months,Month-to-month,Credit card (automatic),89.1,1949.4,0
6713-OKOMC,Female,0,No,No,10,0–12 months,Month-to-month,Mailed check,29.75,301.9,0
7892-POOKP,Female,0,Yes,No,28,25–48 months,Month-to-month,Electronic check,104.8,3046.05,1
6388-TABGU,Male,0,No,Yes,62,49+ months,One year,Bank transfer (automatic),56.15,3487.95,0


Writes the cleaned DataFrame into a Delta table.
overwrite replaces the table so reruns stay consistent

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.telco_churn_customers")
)


In [0]:
%sql
SELECT * 
FROM silver.telco_churn_customers 
LIMIT 10;


customerID,gender,senior_citizen_flag,Partner,Dependents,tenure,tenure_bucket,Contract,PaymentMethod,MonthlyCharges,TotalCharges,churn_flag
7590-VHVEG,Female,0,Yes,No,1,0–12 months,Month-to-month,Electronic check,29.85,29.85,0
5575-GNVDE,Male,0,No,No,34,25–48 months,One year,Mailed check,56.95,1889.5,0
3668-QPYBK,Male,0,No,No,2,0–12 months,Month-to-month,Mailed check,53.85,108.15,1
7795-CFOCW,Male,0,No,No,45,25–48 months,One year,Bank transfer (automatic),42.3,1840.75,0
9237-HQITU,Female,0,No,No,2,0–12 months,Month-to-month,Electronic check,70.7,151.65,1
9305-CDSKC,Female,0,No,No,8,0–12 months,Month-to-month,Electronic check,99.65,820.5,1
1452-KIOVK,Male,0,No,Yes,22,13–24 months,Month-to-month,Credit card (automatic),89.1,1949.4,0
6713-OKOMC,Female,0,No,No,10,0–12 months,Month-to-month,Mailed check,29.75,301.9,0
7892-POOKP,Female,0,Yes,No,28,25–48 months,Month-to-month,Electronic check,104.8,3046.05,1
6388-TABGU,Male,0,No,Yes,62,49+ months,One year,Bank transfer (automatic),56.15,3487.95,0


In [0]:
%sql
DESCRIBE TABLE silver.telco_churn_customers;


col_name,data_type,comment
customerID,string,null
gender,string,null
senior_citizen_flag,bigint,null
Partner,string,null
Dependents,string,null
tenure,bigint,null
tenure_bucket,string,null
Contract,string,null
PaymentMethod,string,null
MonthlyCharges,double,null
